In [ ]:
import os
import random
import math
from collections import Counter
from pprint import pprint
import numpy as np
import pandas as pd
import networkx as nx
import xml.etree.ElementTree as ET
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.utils import from_networkx, to_undirected, negative_sampling
from torch_geometric.loader import NeighborSampler, NeighborLoader
from torch_geometric.nn import GCNConv, SAGEConv, GATConv, GAE, VGAE, NNConv
from torch_geometric.data import Data


In [ ]:
#helper functions

def parse_graphml(file_path):
    """Parse GraphML file and return NetworkX graph"""
    try:
        G = nx.read_graphml(file_path)
        return G
    except Exception:
        return parse_graphml_manual(file_path)

def parse_graphml_manual(file_path):
    """Manual fallback parsing of GraphML"""
    tree = ET.parse(file_path)
    root = tree.getroot()
    G = nx.Graph()

    # Parse keys
    keys = {}
    for key_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}key'):
        keys[key_elem.get('id')] = key_elem.get('attr.name')

    # Parse nodes
    for node_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}node'):
        G.add_node(node_elem.get('id'))

    # Parse edges with attributes
    for edge_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}edge'):
        source = edge_elem.get('source')
        target = edge_elem.get('target')
        edge_attrs = {}

        for data_elem in edge_elem.findall('.//{http://graphml.graphdrawing.org/xmlns}data'):
            key_id = data_elem.get('key')
            attr_name = keys.get(key_id)
            if attr_name:
                try:
                    edge_attrs[attr_name] = float(data_elem.text)
                except:
                    edge_attrs[attr_name] = data_elem.text

        G.add_edge(source, target, **edge_attrs)

    return G

def extract_edge_features(G):
    """Extract edge features as pandas DataFrame"""
    rows = []
    for u, v, data in G.edges(data=True):
        row = {"source": u, "target": v}
        row.update(data)
        rows.append(row)
    return pd.DataFrame(rows)


In [ ]:
#Data loading

data_file_path = "/Users/dannykim/Desktop/vscode/Comp 395/Network-Intrusion-GNN/archive/0.1M-Stratified-Multi.graphml"

# Load data
G = parse_graphml(data_file_path)

print("Number of nodes :", G.number_of_nodes())
print("Number of edges :", G.number_of_edges())

# Extract edge features
edge_df = extract_edge_features(G)

print("\nEdge feature columns :", edge_df.columns.tolist())
print("\nEdge features shape :", edge_df.shape)

print("\nSummary statistics:\n")
print(edge_df.describe())

In [ ]:
#Subgraph of data

random.seed(42)
edges = list(G.edges(data=True))
random.shuffle(edges)

G_sub = nx.Graph()
for u, v, data in edges:
    G_sub.add_edge(u, v, **data)
    if G_sub.number_of_nodes() >= 2000 and G_sub.number_of_edges() >= 8000:
        break

# Keep only largest connected component
largest_cc = max(nx.connected_components(G_sub), key=len)
G_sub = G_sub.subgraph(largest_cc).copy()
print(f"Subgraph: {G_sub.number_of_nodes()} nodes, {G_sub.number_of_edges()} edges")

# Relabel nodes to integers 0..N-1
G_int = nx.convert_node_labels_to_integers(G_sub)

In [ ]:
#dataset split

all_edges = list(G_int.edges())
random.shuffle(all_edges)
n = len(all_edges)

train_edges = all_edges[:int(0.70 * n)]
val_edges   = all_edges[int(0.70 * n):int(0.85 * n)]
test_edges  = all_edges[int(0.85 * n):]

# Training graph (no val/test edges — prevents data leakage)
G_train = nx.Graph()
G_train.add_nodes_from(G_int.nodes())
G_train.add_edges_from(train_edges)

# Negative edges
def generate_negative_edges(G, num_samples, seed=42):
    random.seed(seed)
    neg = []
    nodes = list(G.nodes())
    existing = set(G.edges()) | {(v, u) for u, v in G.edges()}
    while len(neg) < num_samples:
        u, v = random.sample(nodes, 2)
        if (u, v) not in existing:
            neg.append((u, v))
    return neg

neg_train = generate_negative_edges(G_train, len(train_edges))
neg_val   = generate_negative_edges(G_train, len(val_edges),   seed=43)
neg_test  = generate_negative_edges(G_train, len(test_edges),  seed=44)

neg_train_idx = to_idx(neg_train)
neg_val_idx   = to_idx(neg_val)
neg_test_idx  = to_idx(neg_test)

In [ ]:
#data conversion

node2idx = {n: i for i, n in enumerate(G_train.nodes())}

# Edge feature columns to use (drop source, target, and label for now)
EDGE_FEAT_COLS = ['TotPkts', 'TotBytes', 'SrcBytes', 'Dur', 
                  'Proto_encoded', 'Dir_encoded', 'State_encoded']

# Build a lookup from (u,v) -> feature vector using your edge_df
edge_feat_dict = {}
for _, row in edge_df.iterrows():
    u, v = row['source'], row['target']
    feats = row[EDGE_FEAT_COLS].values.astype(float)
    edge_feat_dict[(u, v)] = feats
    edge_feat_dict[(v, u)] = feats  # undirected

# When building your Data object, add edge_attr
edge_list = list(G_train.edges())

edge_index = torch.tensor(
    [[node2idx[u], node2idx[v]] for u, v in edge_list],
    dtype=torch.long
).t().contiguous()

edge_attr = torch.tensor(
    [edge_feat_dict.get((u, v), np.zeros(len(EDGE_FEAT_COLS))) 
     for u, v in edge_list],
    dtype=torch.float
)

def to_idx(edges):
    return [(node2idx[u], node2idx[v]) for u, v in edges
            if u in node2idx and v in node2idx]

train_pos_idx = to_idx(train_edges)
val_pos_idx   = to_idx(val_edges)
test_pos_idx  = to_idx(test_edges)

data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

In [ ]:
#model
from torch_geometric.nn import NNConv

NUM_EDGE_FEATS = len(EDGE_FEAT_COLS)  # 7
NUM_NODE_FEATS = 1                     # degree feature
HIDDEN = 64

class NNConvLinkPredictor(nn.Module):
    def __init__(self, node_in, edge_in, hidden):
        super().__init__()

        # edge_nn output size must equal node_in * hidden
        # because it produces a (node_in x hidden) weight matrix per edge
        self.conv1 = NNConv(
            in_channels=node_in,
            out_channels=hidden,
            nn=nn.Sequential(
                nn.Linear(edge_in, node_in * hidden),
                nn.ReLU()
            )
        )

        # second conv: hidden -> hidden
        self.conv2 = NNConv(
            in_channels=hidden,
            out_channels=hidden,
            nn=nn.Sequential(
                nn.Linear(edge_in, hidden * hidden),
                nn.ReLU()
            )
        )

        # MLP decoder: takes concat of two node embeddings -> edge score
        self.mlp = nn.Sequential(
            nn.Linear(2 * hidden, hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, 1),
            nn.Sigmoid()
        )

    def encode(self, x, edge_index, edge_attr):
        x = F.relu(self.conv1(x, edge_index, edge_attr))
        x = self.conv2(x, edge_index, edge_attr)
        return x

    def decode(self, z, edges):
        u = torch.tensor([e[0] for e in edges], dtype=torch.long)
        v = torch.tensor([e[1] for e in edges], dtype=torch.long)
        return self.mlp(torch.cat([z[u], z[v]], dim=1)).squeeze()

    def forward(self, x, edge_index, edge_attr, edges):
        z = self.encode(x, edge_index, edge_attr)
        return self.decode(z, edges)

In [ ]:
#Evaluation

model = NNConvLinkPredictor(
    node_in=NUM_NODE_FEATS, 
    edge_in=NUM_EDGE_FEATS, 
    hidden=HIDDEN
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

def evaluate(model, data, pos_edges, neg_edges):
    model.eval()
    with torch.no_grad():
        scores = model(data.x, data.edge_index, data.edge_attr, 
                       pos_edges + neg_edges)
    y = [1] * len(pos_edges) + [0] * len(neg_edges)
    return roc_auc_score(y, scores.numpy()), \
           average_precision_score(y, scores.numpy())

for epoch in range(1, 51):
    model.train()
    optimizer.zero_grad()

    all_edges = train_pos_idx + neg_train_idx
    preds = model(data.x, data.edge_index, data.edge_attr, all_edges)
    labels = torch.tensor([1.0] * len(train_pos_idx) + 
                          [0.0] * len(neg_train_idx))
    loss = F.binary_cross_entropy(preds, labels)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        auc, ap = evaluate(model, data, val_pos_idx, neg_val_idx)
        print(f"Epoch {epoch:02d} | Loss: {loss.item():.4f} "
              f"| Val AUC: {auc:.4f} | AP: {ap:.4f}")